# VAE JAX CelebA-HQ Kaggle Pipeline (SiT-B)


In [ ]:
%cd /kaggle/working
!rm -rf RAE
!git clone https://github.com/sontungkieu/RAE
%cd /kaggle/working/RAE
!git checkout jax-vae-sit-celebahq256
!curl -LsSf https://astral.sh/uv/install.sh | sh
!ln -sf /root/.local/bin/uv /usr/local/bin/uv


In [ ]:
import os

os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/.venv"
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"

!uv sync -q
print("Synced the repo dependencies into /tmp/.venv. The package-backed steps below all go through uv run.")


In [ ]:
import os
import pathlib

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["PYOPENGL_PLATFORM"] = "egl"

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    wandb_token = secrets.get_secret("WANDB2")
    hf_token = secrets.get_secret("HF_TOK_WRITE_KAGGLE")

    os.environ["WANDB_API_KEY"] = wandb_token
    os.environ["WANDB_KEY"] = wandb_token
    os.environ["HF_TOKEN"] = hf_token

    netrc = pathlib.Path.home() / ".netrc"
    netrc.write_text(f"machine api.wandb.ai login user password {wandb_token}\n")
    os.chmod(netrc, 0o600)
    print("Loaded Kaggle secrets for wandb and Hugging Face.")
except Exception as exc:
    print(f"Skipping Kaggle secret bootstrap: {exc}")


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

git rev-parse --short HEAD

echo "Using backend-native StabilityVAE; no external RAE decoder download is required."


In [ ]:
from pathlib import Path

repo_root = Path("/kaggle/working/RAE")
hf_cache_dir = Path("/kaggle/working/hf_datasets_cache")
celebahq_root = Path("/kaggle/working/celebahq256_imgfolder")
stage1_cfg_path = repo_root / "configs" / "stage1" / "pretrained" / "CelebAHQ256_StabilityVAE_jax.yaml"
stage2_cfg_path = repo_root / "configs" / "stage2" / "training" / "CelebAHQ256_SiT-B_StabilityVAE_jax.yaml"
fid_stats_path = Path("/kaggle/working/celebahq256_val_fid_stats.pkl")
stage1_single_recon_path = Path("/kaggle/working/celebahq256_vae_stage1_single_recon.png")
stage1_recon_dir = Path("/kaggle/working/celebahq256_vae_stage1_recon_val")
stage2_results_dir = Path("/kaggle/working/results_jax")

recon_batch_size = 8
recon_num_workers = 8
recon_limit = 2048  # đặt None hoặc bỏ --limit ở cell dưới nếu muốn export toàn bộ val split

print("repo_root:", repo_root)
print("hf_cache_dir:", hf_cache_dir)
print("celebahq_root:", celebahq_root)
print("stage1_cfg_path:", stage1_cfg_path)
print("stage2_cfg_path:", stage2_cfg_path)
print("All package-backed cells below use uv run against the synced /tmp/.venv environment.")


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src_jax/export_celebahq_hf.py \
  --dataset eurecom-ds/celeba-hq-256 \
  --cache-dir /kaggle/working/hf_datasets_cache \
  --output /kaggle/working/celebahq256_imgfolder

uv run python - <<'PYSUM'
import json
from pathlib import Path

summary_path = Path("/kaggle/working/celebahq256_imgfolder/hf_export_summary.json")
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
PYSUM


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python - <<'PYCFG'
from pathlib import Path
import textwrap

repo_root = Path("/kaggle/working/RAE")
celebahq_root = Path("/kaggle/working/celebahq256_imgfolder")
stage1_cfg_path = repo_root / "configs" / "stage1" / "pretrained" / "CelebAHQ256_StabilityVAE_jax.yaml"
stage2_cfg_path = repo_root / "configs" / "stage2" / "training" / "CelebAHQ256_SiT-B_StabilityVAE_jax.yaml"
fid_stats_path = Path("/kaggle/working/celebahq256_val_fid_stats.pkl")

stage1_cfg_text = textwrap.dedent(
    """
    stage_1:
      target: stage1.StabilityVAE
      params:
        sample_size: 256
        latent_channels: 4
        downsample_factor: 8
        raw_mean: [0.865, -0.278, 0.216, 0.374]
        raw_std: [4.86, 5.32, 3.94, 3.99]
        final_mean: 0.0
        final_std: 0.5
    """
).strip() + "\n"

stage2_cfg_text = textwrap.dedent(
    f"""
    stage_1:
      target: stage1.StabilityVAE
      ckpt: null
      params:
        sample_size: 256
        latent_channels: 4
        downsample_factor: 8
        raw_mean: [0.865, -0.278, 0.216, 0.374]
        raw_std: [4.86, 5.32, 3.94, 3.99]
        final_mean: 0.0
        final_std: 0.5

    stage_2:
      target: stage2.models.SiT.SiT
      ckpt: null
      params:
        input_size: 32
        patch_size: 2
        in_channels: 4
        hidden_size: 768
        depth: 12
        num_heads: 12
        mlp_ratio: 4.0
        class_dropout_prob: 0.0
        num_classes: 1
        use_qknorm: false
        use_swiglu: true
        use_rope: true
        use_rmsnorm: true
        wo_shift: false

    transport:
      params:
        path_type: 'Linear'
        prediction: 'velocity'
        loss_weight: null
        time_dist_type: 'uniform'

    sampler:
      mode: ODE
      params:
        sampling_method: 'euler'
        num_steps: 50
        atol: 1.0e-6
        rtol: 1.0e-3
        reverse: false

    guidance:
      method: 'cfg'
      scale: 1.0
      t_min: 0.0
      t_max: 1.0

    misc:
      latent_size: [4, 32, 32]
      num_classes: 1
      time_dist_shift_dim: 4096
      time_dist_shift_base: 4096

    eval:
      data_path: '{(celebahq_root / "val").as_posix()}'
      eval_every: 5000
      batch_size: 4
      num_workers: 0
      prefetch_factor: 2
      random_flip: false
      max_batches: 32
      eval_model: false
      fid_ref: '{fid_stats_path.as_posix()}'
      fid_every: 5000
      fid_num_samples: 4096
      fid_per_proc_batch_size: 4
      fid_batch_size: 128

    training:
      global_seed: 0
      epochs: 200
      global_batch_size: 64
      grad_accum_steps: 2
      ema_decay: 0.9995
      num_workers: 8
      prefetch_factor: 4
      random_flip: true
      log_every: 10
      ckpt_every: 5000
      sample_every: 5000
      base_lr: 0.0001
      final_lr: 0.00001
      beta: [0.9, 0.95]
      wd: 0.0
      schedule_type: 'linear'
      decay_start_epoch: 150
      decay_end_epoch: 200
      clip_grad: 1.0
    """
).strip() + "\n"

stage1_cfg_path.parent.mkdir(parents=True, exist_ok=True)
stage2_cfg_path.parent.mkdir(parents=True, exist_ok=True)
stage1_cfg_path.write_text(stage1_cfg_text, encoding="utf-8")
stage2_cfg_path.write_text(stage2_cfg_text, encoding="utf-8")

print(f"Wrote {stage1_cfg_path}")
print(f"Wrote {stage2_cfg_path}")
PYCFG


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python - <<'PYVIEW'
from pathlib import Path

for path in [
    Path("/kaggle/working/RAE/configs/stage1/pretrained/CelebAHQ256_StabilityVAE_jax.yaml"),
    Path("/kaggle/working/RAE/configs/stage2/training/CelebAHQ256_SiT-B_StabilityVAE_jax.yaml"),
]:
    print(f"=== {path} ===")
    print(path.read_text())
PYVIEW


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python - <<'PYCHECK'
from pathlib import Path

sample_image_path = next(path for path in (Path("/kaggle/working/celebahq256_imgfolder") / "val" / "face").iterdir() if path.is_file())
print("sample image:", sample_image_path)
print("No bootstrap latent-stat pass is required for the StabilityVAE flow.")
PYCHECK


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

sample_image=$(find /kaggle/working/celebahq256_imgfolder/val/face -type f -print -quit)
[ -n "${sample_image}" ]

uv run python src_jax/stage1_sample.py \
  --config configs/stage1/pretrained/CelebAHQ256_StabilityVAE_jax.yaml \
  --image "${sample_image}" \
  --output /kaggle/working/celebahq256_vae_stage1_single_recon.png


In [ ]:
display(DisplayImage(filename=stage1_single_recon_path))


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src_jax/reconstruct_folder.py \
  --config configs/stage1/pretrained/CelebAHQ256_StabilityVAE_jax.yaml \
  --input /kaggle/working/celebahq256_imgfolder/val \
  --output-dir /kaggle/working/celebahq256_vae_stage1_recon_val \
  --batch-size 8 \
  --num-workers 8 \
  --limit 2048


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src_jax/build_fid_stats.py \
  --input /kaggle/working/celebahq256_imgfolder/val \
  --output /kaggle/working/celebahq256_val_fid_stats.pkl \
  --image-size 256 \
  --batch-size 64 \
  --num-workers 8


## Optional: launch Stage 2 JAX VAE + SiT-B training on Kaggle GPU


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

export ENTITY="<wandb_entity>"
export PROJECT="vae-jax-celebahq256-$(TZ=Asia/Bangkok date +%Y%m%d-%H%M%S)"

uv run python src_jax/train.py \
  --config configs/stage2/training/CelebAHQ256_SiT-B_StabilityVAE_jax.yaml \
  --data-path /kaggle/working/celebahq256_imgfolder \
  --results-dir /kaggle/working/results_jax \
  --precision bf16 \
  --set training.num_workers=16 \
  --set training.prefetch_factor=4 \
  --set training.log_rae_latent_stats=true \
  --set training.log_activation_stats=true \
  --set eval.prefetch_factor=4 \
  --wandb
